# 01 - Data Cleaning: Food Delivery Analysis

**Dataset:** Food Delivery Time dataset (`train.csv`, 45,593 orders, 20 columns)
**Goal:** Clean the raw file, create new features, and save an analysis-ready file to `data/clean/`.

> `test.csv` has no `Time_taken(min)` column and `Sample_Submission.csv` is only a Kaggle submission template, so this project uses **train.csv only**.

## Data dictionary

| Column | Meaning |
|---|---|
| ID | Unique order ID |
| Delivery_person_ID | Rider ID (starts with a city code, e.g. `BANG`, `PUNE`) |
| Delivery_person_Age / Ratings | Rider age and average rating |
| Restaurant_latitude / longitude | Pickup location |
| Delivery_location_latitude / longitude | Drop location |
| Order_Date | Date of the order (dd-mm-yyyy) |
| Time_Orderd / Time_Order_picked | Time order was placed / picked up |
| Weatherconditions | Weather at delivery time |
| Road_traffic_density | Low, Medium, High, Jam |
| Vehicle_condition | Vehicle condition score (0 to 3) |
| Type_of_order | Snack, Meal, Drinks, Buffet |
| Type_of_vehicle | motorcycle, scooter, electric_scooter, bicycle |
| multiple_deliveries | Other orders the rider carried at the same time |
| Festival | Was it a festival day (Yes/No) |
| City | City type: Metropolitian, Urban, Semi-Urban |
| Time_taken(min) | **Target:** delivery time in minutes (stored as text like `(min) 24`) |

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/train.csv")
print(df.shape)
df.head()

(45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## 1. First inspection

In [2]:
df.info()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs :", df['ID'].duplicated().sum())
print("Real NaN values:", df.isna().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  object 
 1   Delivery_person_ID           45593 non-null  object 
 2   Delivery_person_Age          45593 non-null  object 
 3   Delivery_person_Ratings      45593 non-null  object 
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  object 
 9   Time_Orderd                  45593 non-null  object 
 10  Time_Order_picked            45593 non-null  object 
 11  Weatherconditions            45593 non-null  object 
 12  Road_traffic_density         45593 non-null  object 
 13  Vehicle_conditio

**Observation:** `isna()` finds nothing, but several columns contain the text `"NaN "` (with a trailing space). Pandas treats it as normal text, so the missing values are hidden. Let's count them.

In [3]:
hidden = {}
for c in df.select_dtypes(include=['object', 'string']).columns:
    n = (df[c].astype(str).str.strip() == 'NaN').sum()
    if n:
        hidden[c] = n
pd.Series(hidden, name="hidden NaN count")

Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
Name: hidden NaN count, dtype: int64

## 2. Clean text and missing markers

In [4]:
text_cols = df.select_dtypes(include=['object', 'string']).columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip().replace({'NaN': np.nan, 'nan': np.nan})

df.isna().sum()[df.isna().sum() > 0]

Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
dtype: int64

## 3. Rename columns and fix data types

In [5]:
df = df.rename(columns={'Time_Orderd': 'time_ordered', 'Time_taken(min)': 'time_taken_min'})
df.columns = df.columns.str.lower()

df['delivery_person_age']     = pd.to_numeric(df['delivery_person_age'])
df['delivery_person_ratings'] = pd.to_numeric(df['delivery_person_ratings'])
df['multiple_deliveries']     = pd.to_numeric(df['multiple_deliveries'])

# '(min) 24' -> 24
df['time_taken_min'] = df['time_taken_min'].str.extract(r'(\d+)').astype(int)

# 'conditions Sunny' -> 'Sunny'
df['weatherconditions'] = df['weatherconditions'].str.replace('conditions ', '', regex=False)

df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y')
df.dtypes

id                                     object
delivery_person_id                     object
delivery_person_age                   float64
delivery_person_ratings               float64
restaurant_latitude                   float64
restaurant_longitude                  float64
delivery_location_latitude            float64
delivery_location_longitude           float64
order_date                     datetime64[ns]
time_ordered                           object
time_order_picked                      object
weatherconditions                      object
road_traffic_density                   object
vehicle_condition                       int64
type_of_order                          object
type_of_vehicle                        object
multiple_deliveries                   float64
festival                               object
city                                   object
time_taken_min                          int64
dtype: object

## 4. Fix invalid values

Checks on the raw data showed:
- Some **ratings are 6.0** (the scale ends at 5)
- Some **riders are aged 15** (below the legal working age)
- **Restaurant coordinates are negative** in some rows (India has positive lat/long)
- **3,640 rows have restaurant lat/long = 0**, which is a missing value, not a real location

In [6]:
print("Ratings > 5 :", (df['delivery_person_ratings'] > 5).sum())
print("Age < 18    :", (df['delivery_person_age'] < 18).sum())
print("Negative restaurant lat/long:", (df['restaurant_latitude'] < 0).sum())
print("Zero restaurant coordinates :", ((df['restaurant_latitude'] == 0) & (df['restaurant_longitude'] == 0)).sum())

df.loc[df['delivery_person_ratings'] > 5, 'delivery_person_ratings'] = np.nan
df.loc[df['delivery_person_age'] < 18, 'delivery_person_age'] = np.nan

# negative coordinates are sign errors, so take the absolute value
df['restaurant_latitude']  = df['restaurant_latitude'].abs()
df['restaurant_longitude'] = df['restaurant_longitude'].abs()

# zero coordinates = missing
zero = (df['restaurant_latitude'] == 0) & (df['restaurant_longitude'] == 0)
df.loc[zero, ['restaurant_latitude', 'restaurant_longitude']] = np.nan

Ratings > 5 : 53
Age < 18    : 38
Negative restaurant lat/long: 431
Zero restaurant coordinates : 3640


## 5. Feature engineering

**Distance** is the most important new column. It is calculated from the coordinates with the Haversine formula (straight-line km between pickup and drop).

In [7]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['restaurant_latitude'], df['restaurant_longitude'],
                              df['delivery_location_latitude'], df['delivery_location_longitude'])
df['distance_km'].describe().round(2)

count    41953.00
mean         9.72
std          5.60
min          1.47
25%          4.66
50%          9.19
75%         13.68
max         20.97
Name: distance_km, dtype: float64

In [8]:
# Time features. If time_ordered is missing, use the pickup time (it is only minutes later)
ordered_hour = pd.to_datetime(df['time_ordered'],      format='%H:%M:%S', errors='coerce').dt.hour
picked_hour  = pd.to_datetime(df['time_order_picked'], format='%H:%M:%S', errors='coerce').dt.hour
df['order_hour'] = ordered_hour.fillna(picked_hour)

df['day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend']  = df['day_of_week'].isin(['Saturday', 'Sunday'])
df['month']       = df['order_date'].dt.month_name()

df['time_slot'] = pd.cut(df['order_hour'], bins=[0, 11, 16, 21, 24],
                         labels=['Morning', 'Afternoon', 'Evening', 'Night'], right=False)

# Rider city code from the rider ID, e.g. 'BANGRES18DEL02' -> 'BANG'
df['city_code'] = df['delivery_person_id'].str.extract(r'^([A-Z]+?)RES')

# Late delivery = slower than 75% of all orders (justify this threshold in your README)
late_cutoff = df['time_taken_min'].quantile(0.75)
df['is_late'] = df['time_taken_min'] > late_cutoff
print("Late cutoff (min):", late_cutoff, "| share of late orders:", df['is_late'].mean().round(3))

Late cutoff (min): 32.0 | share of late orders: 0.247


## 6. Handle remaining missing values

| Column | Decision | Reason |
|---|---|---|
| age, ratings | Fill with **median** | Numeric, and the median is not pulled by extreme values |
| weather, traffic, festival, city | Fill with **"Unknown"** | Don't guess categories; keep the rows and show the gap |
| multiple_deliveries | Fill with **mode** | Discrete count, most orders carry 1 |
| restaurant coordinates, distance | **Leave as NaN** | Can't be guessed; exclude these rows only when analysing distance |

In [9]:
df['delivery_person_age']     = df['delivery_person_age'].fillna(df['delivery_person_age'].median())
df['delivery_person_ratings'] = df['delivery_person_ratings'].fillna(df['delivery_person_ratings'].median())
df['multiple_deliveries']     = df['multiple_deliveries'].fillna(df['multiple_deliveries'].mode()[0])

for c in ['weatherconditions', 'road_traffic_density', 'festival', 'city']:
    df[c] = df[c].fillna('Unknown')

df.isna().sum()[df.isna().sum() > 0]

restaurant_latitude     3640
restaurant_longitude    3640
time_ordered            1731
distance_km             3640
dtype: int64

## 7. Final checks and save

In [10]:
print(df.shape)
print("Duplicates:", df.duplicated().sum())
df.describe().round(2)

(45593, 28)
Duplicates: 0


,delivery_person_age,delivery_person_ratings,restaurant_latitude,restaurant_longitude,delivery_location_latitude,delivery_location_longitude,order_date,vehicle_condition,multiple_deliveries,time_taken_min,distance_km,order_hour
count,45593.00,45593.00,41953.00,41953.00,45593.00,45593.00,45593,45593.00,45593.00,45593.00,41953.00,45593.00
mean,29.60,4.64,18.91,76.92,17.47,70.85,2022-03-13 16:32:53.987235072,1.02,0.75,26.29,9.72,17.41
min,20.00,1.00,9.96,72.77,0.01,0.01,2022-02-11 00:00:00,0.00,0.00,10.00,1.47,0.00
25%,25.00,4.60,12.99,73.90,12.99,73.28,2022-03-04 00:00:00,0.00,0.00,19.00,4.66,15.00
50%,30.00,4.70,19.07,76.62,18.63,76.00,2022-03-15 00:00:00,1.00,1.00,26.00,9.19,19.00
75%,34.00,4.80,22.75,78.37,22.79,78.11,2022-03-27 00:00:00,2.00,1.00,32.00,13.68,21.00
max,50.00,5.00,30.91,88.43,31.05,88.56,2022-04-06 00:00:00,3.00,3.00,54.00,20.97,23.00
std,5.68,0.32,5.47,3.50,7.34,21.12,NaN,0.84,0.57,9.38,5.60,4.85


In [11]:
df.to_csv("../data/clean/train_clean.csv", index=False)
print("Saved to data/clean/train_clean.csv")

Saved to data/clean/train_clean.csv
